## Notebook setup

In [1]:
%run "packages.ipynb"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Read data

In [14]:
raw_data = pd.read_csv("data/corona_tested_individuals_ver_006.english.csv")
df = raw_data.copy()

C:\Users\NM2.ZVLKND36\AppData\Local\Temp\ipykernel_24204\2529642436.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv("data/corona_tested_individuals_ver_006.english.csv")


## Pre Processing

### Define outcome variable

Tested individuals do not represent general population - only test if you have symptoms or maybe travelling . Test is therefore "the probability of a positive test among people tested" - not if they have covid!


In [ ]:
print('Data rows before:', len(df))

df_1 = df[df["corona_result"] != "other"].copy()

df_1["corona_result"] = (df_1["corona_result"] == "positive").astype(int)

print('Data rows after:', len(df_1))

df_1.head()

Data rows before: 278848
Data rows after: 274956


,test_date,cough,fever,sore_throat,shortness_of_breath,head_ache,corona_result,age_60_and_above,gender,test_indication
0,2020-04-30,0.0,0.0,0.0,0.0,0.0,0,NaN,female,Other
1,2020-04-30,1.0,0.0,0.0,0.0,0.0,0,NaN,female,Other
2,2020-04-30,0.0,1.0,0.0,0.0,0.0,0,NaN,male,Other
3,2020-04-30,1.0,0.0,0.0,0.0,0.0,0,NaN,female,Other
4,2020-04-30,1.0,0.0,0.0,0.0,0.0,0,NaN,male,Other


### Show dataset balance

In [ ]:
# Class counts
print(df_1["corona_result"].value_counts())

# Class percentages
print(df_1["corona_result"].value_counts(normalize=True) * 100)

corona_result
0    260227
1     14729
Name: count, dtype: int64
corona_result
0    94.643143
1     5.356857
Name: proportion, dtype: float64


### Investigate missing data

In [ ]:
# Missing data summary
missing = pd.DataFrame({
    "missing_n": df_1.isna().sum(),
    "missing_pct": df_1.isna().mean() * 100
})

missing = missing.sort_values("missing_pct", ascending=False)

missing

,missing_n,missing_pct
age_60_and_above,125664,45.703313
gender,19045,6.926563
fever,252,0.091651
cough,252,0.091651
head_ache,1,0.000364
sore_throat,1,0.000364
shortness_of_breath,1,0.000364
test_date,0,0.000000
corona_result,0,0.000000
test_indication,0,0.000000


In [33]:
df_2 = df_1.copy()

df_2["age_60_and_above"] = (
    df_2["age_60_and_above"]
    .fillna("unknown")
    .astype("category")
)

df_2["gender"] = (
    df_2["gender"]
    .fillna("unknown")
    .astype("category")
)

print('Data rows before:', len(df_1))

df_2 = df_2.dropna(
    subset=["sore_throat", "head_ache", "cough", "fever", "shortness_of_breath"]
)

print('Data rows after:', len(df_2))

df_2.head()

Data rows before: 274956
Data rows after: 274702


,test_date,cough,fever,sore_throat,shortness_of_breath,head_ache,corona_result,age_60_and_above,gender,test_indication
0,2020-04-30,0.0,0.0,0.0,0.0,0.0,0,unknown,female,Other
1,2020-04-30,1.0,0.0,0.0,0.0,0.0,0,unknown,female,Other
2,2020-04-30,0.0,1.0,0.0,0.0,0.0,0,unknown,male,Other
3,2020-04-30,1.0,0.0,0.0,0.0,0.0,0,unknown,female,Other
4,2020-04-30,1.0,0.0,0.0,0.0,0.0,0,unknown,male,Other


In [36]:
# Missing data summary
missing = pd.DataFrame({
    "missing_n": df_2.isna().sum(),
    "missing_pct": df_2.isna().mean() * 100
})

missing = missing.sort_values("missing_pct", ascending=False)

missing

,missing_n,missing_pct
test_date,0,0.0
cough,0,0.0
fever,0,0.0
sore_throat,0,0.0
shortness_of_breath,0,0.0
head_ache,0,0.0
corona_result,0,0.0
age_60_and_above,0,0.0
gender,0,0.0
test_indication,0,0.0


### One Hot Encode features

In [35]:
df_3 = df_2.copy()

df_3 = pd.get_dummies(
    df_3,
    columns=["gender", "age_60_and_above", "test_indication"],
    dtype=int
)

df_3.head()

,test_date,cough,fever,sore_throat,shortness_of_breath,head_ache,corona_result,gender_female,gender_male,gender_unknown,age_60_and_above_No,age_60_and_above_Yes,age_60_and_above_unknown,test_indication_Abroad,test_indication_Contact with confirmed,test_indication_Other
0,2020-04-30,0.0,0.0,0.0,0.0,0.0,0,1,0,0,0,0,1,0,0,1
1,2020-04-30,1.0,0.0,0.0,0.0,0.0,0,1,0,0,0,0,1,0,0,1
2,2020-04-30,0.0,1.0,0.0,0.0,0.0,0,0,1,0,0,0,1,0,0,1
3,2020-04-30,1.0,0.0,0.0,0.0,0.0,0,1,0,0,0,0,1,0,0,1
4,2020-04-30,1.0,0.0,0.0,0.0,0.0,0,0,1,0,0,0,1,0,0,1


### Drop date

In [42]:
df_4 = df_3.copy()

df_4 = df_4.drop(columns=["test_date"])

### Ensure format of cols

In [43]:
df_5 = df_4.copy()

df_5 = df_5.astype(int)

## Save new dataset

In [45]:
print(len(df_5))

df_5.to_csv("data/covid_tests_cleaned.csv", index=False)

274702
